# NILM XGBoost Pipeline (Classifier + Regressor)
**Kaggle setup:** Settings -> Accelerator -> GPU (T4 x2 or P100). Required for `device='cuda'` below.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score


## 1. Window shifter and data path

In [ ]:
class WindowShifter:
    @staticmethod
    def shift(df, n):
        keep = df[['Time', 'Aggregate']]
        rest = df.drop(columns=['Time', 'Unix', 'Aggregate'])
        frames = [
            keep.shift(i).rename(columns={'Time': f'Time_t{i}', 'Aggregate': f'Aggregate_t{i}'})
            for i in range(n)
        ]
        return pd.concat(frames + [rest], axis=1).dropna()


if os.path.exists('/kaggle/input'):
    matches = glob.glob('/kaggle/input/nilm-refit-house2-processed/**/House2_full.csv', recursive=True)
    data_path = matches[0] if matches else '/kaggle/input/nilm-refit-house2-processed/House2_full.csv'
else:
    data_path = '../../data/processed_data/House2_full.csv'


## 2. Load, clean, window, and engineer features

In [ ]:
df = pd.read_csv(data_path, encoding='utf-8')

app_cols = [f'Appliance{i}' for i in range(1, 10)]

# fix invalid readings where Aggregate < sum(appliances)
invalid_mask = df['Aggregate'] < df[app_cols].sum(axis=1)
df.loc[invalid_mask, app_cols] = np.nan
df[app_cols] = df[app_cols].interpolate(method='linear').bfill().ffill()
df.loc[invalid_mask, 'Aggregate'] = np.nan
df['Aggregate'] = df['Aggregate'].interpolate(method='linear').bfill().ffill()

df = WindowShifter.shift(df, 50)

aggregate_cols = [c for c in df.columns[:-9] if 'Time' not in c]
df['remain'] = (df['Aggregate_t0'] - df[app_cols].sum(axis=1)).clip(lower=0)

dt = pd.to_datetime(df['Time_t0'], format='%Y-%m-%d %H:%M:%S')
df['dow'] = dt.dt.day_of_week.astype('int8')
df['dom'] = dt.dt.day.astype('int8')
df['hour'] = dt.dt.hour.astype('int8')

# drop Time_t* columns (not used as features) and downcast to float32 to save RAM
df = df.drop(columns=[c for c in df.columns if c.startswith('Time_t')])
for c in aggregate_cols + ['remain'] + app_cols:
    df[c] = df[c].astype('float32')

df.head()


## 3. Feature/target columns and on/off thresholds

In [ ]:
x_cols = ['dow', 'dom', 'hour'] + aggregate_cols
y_cols = app_cols + ['remain']

on_threshold = {
    'Appliance1': 15,   # Fridge-Freezer
    'Appliance2': 20,   # Washing Machine
    'Appliance3': 20,   # Dishwasher
    'Appliance4': 15,   # Television
    'Appliance5': 50,   # Microwave
    'Appliance6': 50,   # Toaster
    'Appliance7': 10,   # Hi-Fi
    'Appliance8': 100,  # Kettle
    'Appliance9': 5,    # Oven Extractor Fan
}

train_mask = ((df['dom'] - 1) // 7 + 1) % 2 == 0
test_mask = ~train_mask


## 4. Classifier: is appliance below the on/off threshold

In [ ]:
y_class = pd.DataFrame({app: df[app] < on_threshold[app] for app in app_cols})

X_train_class, y_train_class = df.loc[train_mask, x_cols], y_class.loc[train_mask]
X_train_class, X_val_class, y_train_class, y_val_class = train_test_split(
    X_train_class, y_train_class, test_size=0.01
)
X_test_class, y_test_class = df.loc[test_mask, x_cols], y_class.loc[test_mask]


In [ ]:
classifier = {}
for app in app_cols:
    clf = xgb.XGBClassifier(
        n_estimators=5000,
        learning_rate=0.05,
        early_stopping_rounds=50,
        tree_method='hist',
        device='cuda',
        eval_metric='logloss',
    )
    clf.fit(X_train_class, y_train_class[app], eval_set=[(X_val_class, y_val_class[app])], verbose=100)
    classifier[app] = clf


In [ ]:
clf_f1 = {app: f1_score(classifier[app].predict(X_test_class), y_test_class[app]) for app in app_cols}
pd.Series(clf_f1, name='F1 (on/off)')


## 5. Regressor: predict power draw per appliance

In [ ]:
X_train, y_train = df.loc[train_mask, x_cols], df.loc[train_mask, y_cols]
X_test, y_test = df.loc[test_mask, x_cols][:70000], df.loc[test_mask, y_cols][:70000]
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.05)


In [ ]:
regressor = {}
for app in app_cols:
    model = xgb.XGBRegressor(
        n_estimators=5000,
        learning_rate=0.05,
        early_stopping_rounds=50,
        tree_method='hist',
        device='cuda',
    )
    model.fit(X_train, y_train[app], eval_set=[(X_val, y_val[app])], verbose=100)
    regressor[app] = model


## 6. Inference and energy-based evaluation

In [ ]:
pred = [regressor[app].predict(X_test) for app in app_cols]
overall_pred = pd.DataFrame(pred).transpose()
overall_pred = overall_pred.clip(lower=0)


In [ ]:
def calculate_nilm_metrics(y_true, y_pred, appliance_names):
    y_t, y_p = np.array(y_true), np.array(y_pred)
    eps = 1e-9
    sum_min = np.minimum(y_p, y_t).sum(axis=0)
    sum_pred, sum_true = y_p.sum(axis=0), y_t.sum(axis=0)

    precision = sum_min / (sum_pred + eps)
    recall = sum_min / (sum_true + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    abs_err = np.abs(y_t - y_p).sum(axis=0)
    nep = abs_err / (sum_true + eps)
    mae = abs_err / y_t.shape[0]

    out = pd.DataFrame({
        'Appliance': appliance_names,
        'Precision (PE)': precision.round(4),
        'Recall (RE)': recall.round(4),
        'F1-Score (FE)': f1.round(4),
        'NEP': nep.round(4),
        'MAE (W)': mae.round(4),
    })
    avg = ['--- AVERAGE ---'] + out.iloc[:, 1:].mean().round(4).tolist()
    out.loc[len(out)] = avg
    return out


metrics_table = calculate_nilm_metrics(
    y_test.reset_index(drop=True).drop(columns=['remain']), overall_pred, app_cols
)
print('XGBoost Regressor rating table:')
display(metrics_table)
